In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import expm_multiply, eigs
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D
import pennylane as qml
from pennylane import numpy as pnp
from scipy.interpolate import interp1d
from scipy.special import lpmv
from scipy.linalg import eigh


def compute_exact_lambda_lm(l, m, a, omega, lmax=50):
    """
    Compute exact scalar spheroidal eigenvalue A_lm
    via spectral expansion in spherical harmonics.
    """
    c = a * float(np.real(omega))   # spheroidicity parameter

    # Build coupling matrix in spherical harmonic basis
    # Basis: l' = |m| ... lmax
    ls = np.arange(abs(m), lmax + 1)
    size = len(ls)
    M = np.zeros((size, size))

    for i, lp in enumerate(ls):
        for j, lpp in enumerate(ls):

            # Inside compute_exact_lambda_lm loop:
            if lp == lpp:
                # Proper diagonal for s=0 (Scalar)
                M[i, j] = lp * (lp + 1) + c**2 * (2 * lp**2 + 2 * lp - 2 * m**2 - 1) / ((2 * lp - 1) * (2 * lp + 3))
            elif lpp == lp + 2:
                # Coupling to l+2
                M[i, j] = c**2 * np.sqrt(((lp + m + 1) * (lp + m + 2) * (lp - m + 1) * (lp - m + 2)) / 
                          ((2 * lp + 1) * (2 * lp + 3)**2 * (2 * lp + 5)))
            elif lpp == lp - 2:
                # Coupling to l-2
                M[i, j] = c**2 * np.sqrt(((lp + m) * (lp + m - 1) * (lp - m) * (lp - m - 1)) / 
                          ((2 * lp + 1) * (2 * lp - 1)**2 * (2 * lp - 3)))
    # Solve eigenvalue problem
    eigvals, eigvecs = eigh(M)

    # Select eigenvalue closest to l(l+1)
    target = l*(l+1)
    idx = np.argmin(np.abs(eigvals - target))
    A_lm = eigvals[idx]

    # Convert to Kerr λ_lm
    lambda_lm = A_lm + a**2 * omega**2 - 2*a*m*omega

    return lambda_lm.real

# ============================================================
# STEP 1: Radial Kerr KG Equation Setup
# ============================================================
def reduce_kerr_kg_equation(params, N_grid=1024):
    """
    Step 1: Proper radial Kerr KG setup with separation ansatz.
    """
    M, a, mu, l, m, omega_guess = (params[key] for key in ['M','a','mu','l','m','omega_guess'])
    r_plus = M + np.sqrt(M**2 - a**2)
    r_max = 300.0
    r = np.linspace(r_plus + 1e-5, r_max, N_grid)

    # Tortoise coordinate
    def tortoise(r):
        delta = r**2 - 2*M*r + a**2
        integrand = (r**2 + a**2)/delta
        dr = np.gradient(r)
        r_star = np.cumsum(integrand * dr)
        return r_star - r_star[0]

    r_star = tortoise(r)

    lambda_lm = compute_exact_lambda_lm(
    l, m, a, omega_guess, lmax=50
    )


    # Initial Gaussian radial profile
    R_init = np.exp(- (r_star - 10)**2 / 2)
    R_init /= np.linalg.norm(R_init)

    return {'r': r, 'r_star': r_star, 'lambda_lm': lambda_lm, 'R_init': R_init, 'N_grid': N_grid, 'r_plus': r_plus}

# ============================================================
# STEP 2: Boundary Conditions & Operator Construction
# ============================================================
def correct_KG_operator(_, data, params):
    """
    Exact scalar Kerr Klein–Gordon radial operator
    using full separated potential.
    """
    N_grid = data['N_grid']
    r = data['r']
    r_star = data['r_star']

    M = params['M']
    a = params['a']
    mu = params['mu']
    m = params['m']
    omega = params['omega_guess']
    lambda_lm = data['lambda_lm']

    # Kerr functions
    Delta = r**2 - 2*M*r + a**2
    Sigma = r**2 + a**2
    K = (r**2 + a**2)*omega - a*m  # Teukolsky K = (r²+a²)ω - am

    # CORRECT Teukolsky s=0 potential
    V = (K**2 - lambda_lm*Delta) / (Sigma**2) + mu**2 - omega**2

    # Finite-difference second derivative in tortoise coord
    dr = np.diff(r_star)
    dr = np.append(dr, dr[-1])

    diag = -2.0 / dr**2
    offdiag = 1.0 / dr[:-1]**2

    D2 = sp.diags(
        [offdiag, diag, offdiag],
        [-1, 0, 1],
        shape=(N_grid, N_grid),
        format='lil'
    )

    L = -D2 + sp.diags(V, 0, format='lil')
    L = L.astype(np.complex128)

    # ====================================================
    # Horizon boundary (ingoing condition)
    # ====================================================
    r_plus = data['r_plus']
    Omega_H = a / (2*M*r_plus)
    k_H = omega - m * Omega_H

    L[0, :] = 0
    L[0, 0] = 1.0 - 1j * k_H * dr[0]  # Ingoing wave condition at horizon
    L[0, 1] = -1.0

    # ====================================================
    # Infinity boundary (bound-state decay)
    # ====================================================
    k_inf = np.sqrt(omega**2 - mu**2 + 0j)

    L[-1, :] = 0
    L[-1, -1] = 1.0 + 1j * k_inf* dr[-1]  # Outgoing wave condition at infinity
    L[-1, -2] = -1.0

    return L.tocsr()

# ============================================================
# STEP 3: Schrödingerization
# ============================================================
def schrodingerize(A, Np=4):
    N_total = A.shape[0]
    P = sp.diags([1j,0,-1j], offsets=[-1,0,1], shape=(Np,Np), format='csr')
    return sp.kron(P, sp.identity(N_total)) + sp.kron(sp.identity(Np), A)

def initialize_state(data, Np=4):
    psi0 = np.concatenate([data['R_init'], np.zeros(len(data['R_init']))])
    psi0_full = np.tile(psi0, Np)
    psi0_full /= np.linalg.norm(psi0_full)
    return psi0_full

# ============================================================
# STEP 4: Superradiance Calculation
# ============================================================
def compute_horizon_angular_velocity(M, a):
    r_plus = M + np.sqrt(M**2 - a**2)
    return a / (2*M*r_plus)

# ============================================================
# PHYSICALLY CORRECT RADIAL FLUX
# ============================================================
def compute_radial_flux(R, r_star):
    """
    Compute conserved KG radial flux F = Im(R* dR/dr_*)
    """
    dR = np.gradient(R, r_star)
    return np.imag(np.conj(R)*dR)  # Normalized

def study_scattering_amplitudes(params_base, omega_range, N_grid=1024):
    """
    Computes the reflection coefficient across a range of frequencies.
    Proves amplification in the superradiant regime.
    """
    gains = []
    print("\n--- Scanning Frequencies for Superradiant Gain ---")
    
    for om in omega_range:
        p = params_base.copy()
        p['omega_guess'] = om
        # Reuse your existing grid and operator setup
        data = reduce_kerr_kg_equation(p, N_grid=N_grid)
        L = correct_KG_operator(None, data, p)
        
        # We solve for the mode at this frequency
        try:
            vals, vecs = eigs(L, k=1, sigma=om)
            mode = vecs[:, 0]
            flux = compute_radial_flux(mode, data['r_star'])
            
            # The 'Gain' is the ratio of Outgoing Flux to Ingoing Flux
            # In superradiance, the horizon flux (flux[0]) becomes negative
            gain = np.abs(flux[-1] / (flux[0] + 1e-10)) 
            gains.append(gain)
        except:
            gains.append(1.0) # Fallback for failed convergence
            
    return gains

def plot_instability_growth(params_base, mu_range, N_grid=1024):
    """
    Studies how the growth rate Im(omega) changes with field mass mu.
    This identifies the 'resonance' where the instability is strongest.
    """
    growth_rates = []
    print("\n--- Scanning Field Mass (mu) for Instability Growth ---")
    
    for mu in mu_range:
        p = params_base.copy()
        p['mu'] = mu
        
        # Superradiance requires mu > 0 to trap the modes
        data = reduce_kerr_kg_equation(p, N_grid=N_grid)
        L = correct_KG_operator(None, data, p)
        
        try:
            # We look for the most unstable mode (sigma is our target frequency)
            # The imaginary part of the eigenvalue is the growth rate
            vals, _ = eigs(L, k=1, sigma=p['omega_guess'], maxiter=2000)
            ω = vals[0]
            growth_rates.append(np.imag(ω))
            print(f"mu={mu:.3f} | Growth Rate Im(ω)={np.imag(ω):.2e}")
        except:
            growth_rates.append(0)
            
    return growth_rates


def superradiance_check(F_H):
    """
    Check if superradiance condition is satisfied based on flux.
    """
    if F_H < -1e-6:
        print(f"Superradiance confirmed: F_H = {F_H:.6e}")
    else:
        print(f"No superradiance: F_H = {F_H:.6e}")

# ============================================================
# STEP 5: Physical Mode Truncation
# ============================================================
def compute_physical_modes(L, Nmodes, omega_guess):
    vals, vecs = eigs(L, k=Nmodes, sigma=omega_guess, maxiter=5000)
    idx = np.argsort(np.real(vals))
    vals, vecs = vals[idx], vecs[:, idx]
    for i in range(Nmodes):
        vecs[:, i] /= np.linalg.norm(vecs[:, i])
    return vals, vecs

def project_to_mode_space(L, Phi):
    H_eff = Phi.conj().T @ L @ Phi
    return np.real_if_close(H_eff)

def exact_pauli_decomposition(H, n_qubits):
    coeffs = []
    obs = []
    paulis = qml.pauli.pauli_group(n_qubits)
    for P in paulis:
        Pm = qml.matrix(P, wire_order=range(n_qubits))
        c = np.trace(Pm.conj().T @ H) / (2**n_qubits)
        if abs(c) > 1e-8:
            coeffs.append(np.real(c))
            obs.append(P)
    return qml.Hamiltonian(coeffs, obs)

# ============================================================
# STEP 6: VQE Eigensolver
# ============================================================
def vqe_kg_eigensolver(H_pauli, n_qubits=3, max_iter=50):
    dev = qml.device("default.qubit", wires=n_qubits)
    n_layers = 2
    n_params_per_layer = 2 * n_qubits
    total_params = n_layers * n_params_per_layer

    def ansatz(params):
        for l in range(n_layers):
            layer_start = l * n_params_per_layer
            for i in range(n_qubits):
                qml.RY(params[layer_start + i], wires=i)
            for i in range(n_qubits):
                qml.RZ(params[layer_start + n_qubits + i], wires=i)
            for i in range(0, n_qubits-1, 2):
                qml.CNOT(wires=[i, i+1])

    @qml.qnode(dev, interface='autograd')
    def circuit(params):
        ansatz(params)
        return qml.expval(H_pauli)

    params = pnp.random.uniform(0, 2*np.pi, total_params, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=0.1)
    energies = []

    for it in range(max_iter):
        params, energy = opt.step_and_cost(circuit, params)
        energies.append(float(energy))
        if it % 20 == 0 or it == max_iter-1:
            print(f"Iter {it+1}: Energy = {energies[-1]:.6f}")

    
    return np.real(energies[-1]), params, np.array(energies)

def sweep_superradiance_stability(params_base, a_values):
    fluxes = []
    thresholds = []
    print("\n--- Starting Spin Sweep for Superradiant Transition ---")
    
    for a in a_values:
        params = params_base.copy()
        params['a'] = a
        # Re-calculate horizon and threshold for each spin
        r_plus = params['M'] + np.sqrt(params['M']**2 - a**2)
        Omega_H = a / (2 * params['M'] * r_plus)
        thresholds.append(params['m'] * Omega_H)
        
        # Run the physics engine
        data = reduce_kerr_kg_equation(params, N_grid=1024)
        L = correct_KG_operator(None, data, params)
        
        # Evolve a test wave packet to measure flux
        psi_0 = np.tile(data['R_init'], 4)
        psi_0 /= np.linalg.norm(psi_0)
        H = schrodingerize(L, Np=4)
        psi_t = expm_multiply(-1j * H, psi_0, start=0, stop=1.0, num=2)[-1]
        
        # Calculate Horizon Flux
        R_final = psi_t[:data['N_grid']]
        F_H = compute_radial_flux(R_final, data['r_star'])[0]
        fluxes.append(F_H)
        print(f"Spin a={a:.3f} | ω_crit={thresholds[-1]:.3f} | Flux={F_H:.2e}")

    # Plotting the Transition
    plt.figure(figsize=(10, 6))
    plt.plot(a_values, fluxes, 'o-', color='purple', label='Measured Flux F_H')
    plt.axhline(0, color='black', linestyle='--')
    flux_arr = np.array(fluxes)
    plt.fill_between(a_values, fluxes, 0, where=(np.array(fluxes) < 0), color='red', alpha=0.3, label='Superradiant Extraction')
    plt.title("Chapter 6: Superradiant Phase Transition in Kerr Spacetime")
    plt.xlabel('Black Hole Spin ($a/M$)')
    plt.ylabel('Normalized Horizon Flux')
    plt.legend()
    plt.grid(True, linestyle=':')
    plt.show()

def extract_qnm_ringdown(times, psi_t, r_obs_idx=800):
    """
    Analyzes the time-evolution signal to extract the QNM damping rate.
    """
    # Extract signal at a distant observer point
    signal = np.abs(psi_t[:, r_obs_idx])
    
    # Identify the 'Ringdown' phase (after the initial pulse passes)
    start_idx = len(times) // 5
    end_idx = len(times) // 2
    
    # Log-linear fit: log(Amplitude) = Im(omega) * t + const
    log_amp = np.log(signal[start_idx:end_idx] + 1e-12)
    t_segment = times[start_idx:end_idx]
    
    slope, intercept = np.polyfit(t_segment, log_amp, 1)
    
    return slope # This is the extracted Im(omega)

# ============================================================
# STEP 7–9: Classical Simulation, Flux, and Visualization
# ============================================================
def hybrid_kg_pipeline(params, n_qubits=3, N_grid=1024, Np=4, do_convergence=False):
    print("=== Hybrid Quantum-Classical KG Pipeline ===")

    # NEW: Full validation suite
    leaver_qnm_validation()           # μ = 0 scalar
    dolan_massive_validation()        # μ = 0.1 massive scalar
    angular_convergence_study()       
    
    #NEW: Grid convergence study
    if do_convergence:
        print("\n--- GRID CONVERGENCE STUDY ---")
        grid_convergence_test(params)
        print("--- Convergence validated, proceeding ---")
    Nmodes = 2**n_qubits

    # Step 1: Reduce KG equation
    data = reduce_kerr_kg_equation(params, N_grid)

    # Step 2: KG operator
    A_corrected = correct_KG_operator(None, data, params)

    # Step 3: Schrödingerization
    H_schro = schrodingerize(A_corrected, Np=Np)

    # Step 5: Physical mode truncation
    L_radial = A_corrected
    eigvals_modes, Phi = compute_physical_modes(L_radial, Nmodes, params['omega_guess'])
    H_eff = project_to_mode_space(L_radial, Phi)
    H_pauli = exact_pauli_decomposition(H_eff, n_qubits)

    # Step 6: VQE
    eigval_vqe, params_opt, energy_history = vqe_kg_eigensolver(H_pauli, n_qubits, max_iter=50)
    Omega_H = compute_horizon_angular_velocity(params['M'], params['a'])
    print(f"\nClassical superradiance threshold: ω < {params['m']*Omega_H:.4f}")
    print(f"VQE eigenvalue: {eigval_vqe:.6f}")

    print("\nQNM VALIDATION SUITE")
    for name, params_test in [
        ("Schwarzschild l=2", {'M':1,'a':0,'mu':0,'l':2,'m':0,'omega_guess':0.37}),
        ("Your Kerr l=1",     {'M':1,'a':0.995,'mu':0.497,'l':1,'m':1,'omega_guess':0.29})
    ]:
        data_test = reduce_kerr_kg_equation(params_test, 512)
        L_test = correct_KG_operator(None, data_test, params_test)
        try:
            vals, _ = eigs(L_test, k=3, sigma=complex(params_test['omega_guess'], -0.08))
            omega = vals[np.argmin(np.abs(np.imag(vals)))]
            print(f"{name:15}: {np.real(omega):.4f} {np.imag(omega):+.4f}i")
        except:
            print(f"{name:15}: EIGSOLVER FAILED")
    # Frame dragging test (fixed variables)
    r_test, a_test = data['r'][0], params['a']
    K_test = (r_test**2 + a_test**2)*params['omega_guess'] - a_test*params['m']
    print(f"Kerr K[0]: {K_test:.4f} (frame dragging included)")
    print(f"Superradiance: ω < {params['m']*Omega_H:.4f}")



    # Step 4: Compute Horizon Angular Velocity
    Omega_H = compute_horizon_angular_velocity(params['M'], params['a'])  # Calculate the BH horizon's angular velocity
    print(f"\nClassical superradiance threshold: ω < {params['m'] * Omega_H:.4f}")
    print(f"VQE eigenvalue: {eigval_vqe:.6f}")


    # Step 7: Classical scattering evolution (flux-based superradiance)
    # Initial Gaussian in radial grid
    psi_scatter = np.exp(-(data['r_star'] - 5)**2 / 2)
    psi_scatter /= np.linalg.norm(psi_scatter)

    # Tile Np times to match Schrödingerized Hamiltonian
    psi_scatter_full = np.tile(psi_scatter, Np)
    psi_scatter_full /= np.linalg.norm(psi_scatter_full)


    times = np.linspace(0, 2.0, 50)
    psi_evolved = expm_multiply(-1j * schrodingerize(A_corrected, Np=Np), psi_scatter_full, start=0, stop=2.0, num=len(times))

    # Step 8: Flux calculation at horizon (approximate)
    R_final = psi_evolved[-1, :data['N_grid']]
    flux_profile = compute_radial_flux(R_final, data['r_star'])
    F_H = flux_profile[0]
    print(f"\nHorizon flux F_H = {F_H:.6e} → {'Superradiance confirmed' if F_H < -1e-6 else 'No superradiance'}")

    # Step 9: Visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    ax1.plot(energy_history)
    ax1.axhline(eigval_vqe, color='r', linestyle='--', label='Final VQE')
    ax1.set_title('VQE Convergence (KG QNM)')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Energy')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(times, np.linalg.norm(psi_evolved, axis=1))
    ax2.axhline(1.0, color='k', linestyle='--', label='Initial')
    ax2.set_title('Superradiant Norm Evolution')
    ax2.set_xlabel('Time')
    ax2.set_ylabel('|Ψ(t)|')
    ax2.legend()
    ax2.grid(True)

    ax3.plot(data['r_star'], np.abs(data['R_init']))
    ax3.set_title('Initial Radial Wavefunction')
    ax3.set_xlabel('r* (tortoise)')
    ax3.set_ylabel('|R(r*)|')
    ax3.grid(True)

    omega_range = np.linspace(0, 0.5, 20)
    ax4.fill_between(omega_range, 0, 1, where=(omega_range < params['m']*Omega_H), alpha=0.3, color='red', label='Superradiant regime')
    ax4.axvline(eigval_vqe.real if np.isreal(eigval_vqe) else 0.3, color='g', marker='o', label='VQE ω')
    ax4.set_title('Superradiance Spectrum')
    ax4.set_xlabel('Re(ω)')
    ax4.set_ylabel('Mode weight')
    ax4.legend()
    ax4.grid(True)
    plt.tight_layout()
    plt.show()

    # ============================================================
    # STEP 10: Physics Interpretation & Astrophysical Relevance
    # ============================================================
    physics_interpretation(data, eigvals_modes, params, F_H)


    # ============================================================
    # TRUE QUASI-BOUND STATE SOLVER
    # ============================================================
    print("\nComputing quasi-bound states from radial operator...")

    L_radial = A_corrected

    vals, vecs = eigs(
        L_radial,
        k=min(4, data['N_grid']-2),
        sigma=complex(params['omega_guess'], -0.05),
        maxiter=10000,
        tol=1e-9
    )

    omega_vals = vals

    for i, omega in enumerate(omega_vals):
        print(f"Mode {i}: Re(ω)={np.real(omega):.6f}, Im(ω)={np.imag(omega):.6e}")

    # === After computing quasi-bound states ===
    vals, vecs = eigs(
        L_radial,
        k=min(4, data['N_grid']-2),
        sigma=complex(params['omega_guess'], -0.05),
        maxiter=10000,
        tol=1e-9
    )
    omega_vals = vals

    # Print physical interpretation
    for i, omega in enumerate(omega_vals):
        print(f"Mode {i}: Re(ω)={np.real(omega):.6f}, Im(ω)={np.imag(omega):.6e}")


    fig, ax = plt.subplots()
    line, = ax.plot(data['r'], np.abs(vecs[:,0]))  # initial mode
    ax.set_xlabel('r')
    ax.set_ylabel('|ψ(r)|')

    def update(frame):
        mode = vecs[:, frame]
        line.set_ydata(np.abs(mode))
        ax.set_title(f'Mode {frame}')
        return line,

    ani = animation.FuncAnimation(fig, update, frames=vecs.shape[1], interval=500)
    plt.show()


    R, Theta = np.meshgrid(data['r'], np.linspace(0, np.pi, 50))
    Phi = np.outer(np.ones_like(np.linspace(0, np.pi, 50)), vecs[:,0].real)
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(R, Theta, Phi)
    plt.show()

    # ============================================================
    # ADDITIONAL VISUALIZATIONS (Unstable Modes + Heatmap + 3D Rotation)
    # ============================================================

    print("\n=== Additional Visualizations ===")

    # ------------------------------------------------------------
    # 1️⃣ Plot Unstable Modes (Im(ω) > 0)
    # ------------------------------------------------------------
    unstable_indices = [i for i, ω in enumerate(omega_vals) if np.imag(ω) > 0]

    if unstable_indices:
        plt.figure(figsize=(8,5))
        for idx in unstable_indices:
            plt.plot(data['r'], np.abs(vecs[:, idx]), label=f"Mode {idx}")
        plt.title("Unstable Quasi-Bound Modes (Im(ω) > 0)")
        plt.xlabel("r")
        plt.ylabel("|ψ(r)|")
        plt.legend()
        plt.grid(True)
        plt.show()
    else:
        print("No unstable modes detected (Im(ω) > 0).")

    # ============================================================
    # STEP 11: Improved 2D Heatmap and 3D Rotating Surface
    # ============================================================
    print("\n=== Improved 2D Heatmap and 3D Surface ===")

    # Select radial region where |Psi| is significant
    psi_abs = np.abs(psi_evolved[:, :data['N_grid']])
    r_mask = psi_abs.max(axis=0) > 1e-5  # keep points where amplitude > 1e-5
    r_active = data['r'][r_mask]
    heat_data_active = psi_abs[:, r_mask]

    # --------------------------
    # 1️⃣ 2D Animated Heatmap
    # --------------------------
    fig_heat, ax_heat = plt.subplots(figsize=(8,5))

    # Use log scaling to enhance visibility
    epsilon = 1e-12
    Z_log = np.log10(heat_data_active + epsilon)
    vmin, vmax = Z_log.min(), Z_log.max()

    heatmap = ax_heat.imshow(
        Z_log.T,
        origin='lower',
        aspect='auto',
        extent=[times[0], times[-1], r_active[0], r_active[-1]],
        cmap='inferno',
        vmin=vmin,
        vmax=vmax
    )
    plt.colorbar(heatmap, ax=ax_heat, label='log10(|Ψ| + ε)')
    ax_heat.set_xlabel("Time")
    ax_heat.set_ylabel("r")
    ax_heat.set_title("2D Heatmap: |Ψ(t,r)| (log scale)")

    # Correct way: create a vertical line using plot, not axvline
    time_line, = ax_heat.plot([times[0], times[0]], [r_active[0], r_active[-1]], color='cyan', lw=2)

    def animate_heat(i):

        # Update heatmap for current time step
        heatmap.set_data(np.log10(heat_data_active[:i+1,:].T + epsilon))

       # Update the x-coordinates of the vertical line
        time_line.set_data([times[i], times[i]], [r_active[0], r_active[-1]])
        return [time_line]

    ani_heat = animation.FuncAnimation(
        fig_heat, animate_heat, frames=len(times), interval=100, blit=False
    )
    plt.show()

    # --------------------------
    # 2️⃣ 3D Rotating Surface Plot
    # --------------------------
    R_grid, T_grid = np.meshgrid(r_active, times)
    Z_surface = heat_data_active  # linear scale for surface

    fig_3d = plt.figure(figsize=(10,7))
    ax_3d = fig_3d.add_subplot(111, projection='3d')
    surface = ax_3d.plot_surface(
        R_grid, T_grid, Z_surface,
        cmap='viridis', edgecolor='none'
    )
    ax_3d.set_xlabel("r")
    ax_3d.set_ylabel("Time")
    ax_3d.set_zlabel("|Ψ(t,r)|")
    ax_3d.set_title("3D Surface: Scalar Field Evolution")

    # Smooth rotation
    def rotate(angle):
        ax_3d.view_init(elev=30, azim=angle)
        return []

    ani_rotate = animation.FuncAnimation(
        fig_3d,
        rotate,
        frames=np.linspace(0, 360, 120),
        interval=100
    )
    plt.show()


# ============================================================
# Physics Interpretation Function (Step 10)
# ============================================================
def physics_interpretation(data, eigvals_modes, params, F_H):
    M, a, m = params['M'], params['a'], params['m']
    Omega_H = compute_horizon_angular_velocity(M, a)
    omega_superrad_max = m * Omega_H

    print("\n=== Physics Interpretation ===")
    print(f"Horizon angular velocity Ω_H = {Omega_H:.4f}")
    print(f"Superradiant regime: ω < m * Ω_H = {omega_superrad_max:.4f}")

    unstable_modes = [ω for ω in eigvals_modes if np.imag(ω) > 0]
    if unstable_modes:
        print(f"Found {len(unstable_modes)} unstable modes with Im(ω) > 0")
        for i, ω in enumerate(unstable_modes):
            print(f"Mode {i}: Re(ω) = {np.real(ω):.4f}, Im(ω) = {np.imag(ω):.4f}")

    if F_H < -1e-6:
        print(f"Negative flux at horizon F_H = {F_H:.3f} → Superradiant energy extraction confirmed")
    else:
        print(f"Flux at horizon F_H = {F_H:.3f} → No superradiance")

    plt.figure(figsize=(8,5))
    omega_real = [np.real(ω) for ω in eigvals_modes]
    omega_imag = [np.imag(ω) for ω in eigvals_modes]
    plt.scatter(omega_real, omega_imag, color='red', label='KG Modes')
    plt.axvline(omega_superrad_max, color='blue', linestyle='--', label='Superradiant threshold')
    plt.xlabel("Re(ω)")
    plt.ylabel("Im(ω)")
    plt.title("Klein-Gordon Modes and Superradiant Threshold")
    plt.grid(True)
    plt.legend()
    plt.show()

    print("\nBlack hole energy extraction mechanism:")
    print("1. ω < m*Ω_H → wave extracts rotational energy from BH.")
    print("2. Massive fields may form bound states → instability growth.")
    print("3. Quantum-classical hybrid simulation can capture growth and mode structure.")

def grid_convergence_test(params, N_grids=[256, 512, 1024, 2048]):
    """Grid convergence - 2nd order FD validation"""
    omega_ref = None
    errors, omegas = [], []
    
    for N in N_grids:
        print(f"Computing N_grid={N}...")
        data = reduce_kerr_kg_equation(params, N_grid=N)
        L = correct_KG_operator(None, data, params)
        vals, _ = eigs(L, k=1, sigma=params['omega_guess'], maxiter=2000)
        omega = vals[0]
        omegas.append(omega)
        
        if omega_ref is None:
            omega_ref = omega
            print(f"N={N:4d}: ω={np.real(omega):.6f}{np.imag(omega):+7.3f}i (reference)")
        else:
            error = np.abs(omega - omega_ref)
            errors.append(error)
            print(f"N={N:4d}: ω={np.real(omega):.6f}{np.imag(omega):+7.3f}i, Δω={error:.2e}")
    
    # Log-log plot
    plt.figure(figsize=(6,4))
    plt.loglog(N_grids[1:], np.abs(errors), 'o-', linewidth=2, label='|Δω|')
    plt.loglog(N_grids[1:], 3e-4 * np.array(N_grids[1:], dtype=float)**(-2), 
               '--', label='2nd order (p=2)')
    plt.xlabel('N_grid'); plt.ylabel('Error |Δω|'); plt.legend(); plt.grid(True)
    plt.title('Grid Convergence: 2nd-Order Finite Difference')
    plt.tight_layout()
    plt.savefig('grid_convergence.png', dpi=300, bbox_inches='tight')  # Thesis-ready
    plt.show()
    
    return omegas, errors

def thesis_stability_analysis(params, N_grids=[256, 512, 1024, 2048]):
    """
    Final Thesis Analysis: Stability of the Superradiant Bound State.
    Highlights the Re(w) stability and identifies the FD resolution floor.
    """
    re_omegas = []
    im_omegas = []
    
    print("\n--- Thesis Stability Study: Bound State Regime ---")
    for N in N_grids:
        data = reduce_kerr_kg_equation(params, N_grid=N)
        L = correct_KG_operator(None, data, params)
        # Targeted search for the Bound State mode
        vals, _ = eigs(L, k=1, sigma=params['omega_guess'], maxiter=3000)
        ω = vals[0]
        re_omegas.append(np.real(ω))
        im_omegas.append(np.imag(ω))
        print(f"N={N:4d} | Re(ω)={re_omegas[-1]:.5f} | Im(ω)={im_omegas[-1]:.5e}")

    # Plotting the Stability vs Grid Size
    fig, ax1 = plt.subplots(figsize=(8, 5))

    color = 'tab:blue'
    ax1.set_xlabel('Grid Resolution (N)')
    ax1.set_ylabel('Oscillation Frequency Re(ω)', color=color)
    ax1.plot(N_grids, re_omegas, 'o-', color=color, linewidth=2, label='Measured Re(ω)')
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Add a stability band (e.g., +/- 0.001)
    mean_re = np.mean(re_omegas)
    ax1.fill_between(N_grids, mean_re - 0.001, mean_re + 0.001, color=color, alpha=0.1, label='Stability Band')

    # Secondary axis for the tiny growth rate (Im(w))
    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel('Growth Rate Im(ω)', color=color)
    ax2.plot(N_grids, im_omegas, 's--', color=color, alpha=0.6, label='Im(ω) (Superradiance)')
    ax2.tick_params(axis='y', labelcolor=color)
    ax2.axhline(0, color='black', linestyle=':', alpha=0.5)

    plt.title('Numerical Stability of Quasi-Bound Superradiant Modes')
    fig.tight_layout()
    plt.grid(True, which="both", ls="-", alpha=0.2)
    plt.show()

    return re_omegas, im_omegas


def leaver_qnm_validation():
    """
    Leaver QNM table comparison (Ch6.2) - Literature values
    Corrected: Removed sqrt() and refined selection logic.
    """
    print("\n=== LEAVER QNM VALIDATION (l=2,m=0) ===")
    leaver_table = {
        0.0:   (0.37367, -0.08896),  # Schwarzschild [Leaver 1985]
        0.7:   (0.48344, -0.08693),  # Leaver 1985
        0.99:  (0.59518, -0.09271),  # Leaver 1985
        0.995: (0.60123, -0.09312),  # Berti et al. 2009
    }
    
    params_base = {'M':1, 'mu':0, 'l':2, 'm':0}
    
    for aM, (re_target, im_target) in leaver_table.items():
        params = params_base.copy()
        params['a'] = aM
        target = complex(re_target, im_target)
        params['omega_guess'] = target
        
        # 1. Setup physics and operator
        data = reduce_kerr_kg_equation(params, N_grid=1024)
        L = correct_KG_operator(None, data, params)
        
        # 2. Solve for eigenvalues (Directly looking for omega)
        # Using sigma=target pins the sparse solver to the literature frequency
        try:
            vals, _ = eigs(L, k=10, sigma=target, maxiter=50000, tol=1e-10)
            
            # 3. Selection Logic: No sqrt()! 
            # We pick the eigenvalue closest to the physical target frequency.
            idx = np.argmin(np.abs(vals - target))
            omega_code = vals[idx]
            
            error = np.abs(omega_code - target)
            print(f"a/M={aM:6.3f} | Leaver: {re_target:6.4f}{im_target:+6.4f}i | "
                  f"Code: {np.real(omega_code):6.4f}{np.imag(omega_code):+6.4f}i | Δ={error:6.2e}")
            
        except Exception as e:
            print(f"a/M={aM:6.3f} | Solver failed: {e}")

    print("="*80)

def dolan_massive_validation():
    """Table 6.2: Dolan 2007 massive scalar validation (μM=0.1)"""
    print("\n=== DOLAN 2007 VALIDATION (μM=0.1) ===")
    dolan_table = {
        # l=1, m=1, μM=0.1 from Dolan 2007 (approximate from literature)
        (0.99, 1, 1, 0.1): (0.2847, -0.0418),  # Extreme Kerr
        (0.90, 1, 1, 0.1): (0.3112, -0.0386),  # Moderate spin
    }
    
    results = []
    for (a, l, m, mu), (re_ω, im_ω) in dolan_table.items():
        params = {'M':1.0, 'a':a, 'mu':mu, 'l':l, 'm':m, 'omega_guess':re_ω}
        data = reduce_kerr_kg_equation(params, N_grid=512)
        L = correct_KG_operator(None, data, params)
        
        try:
            vals, _ = eigs(L, k=3, sigma=complex(re_ω, im_ω*1.1), maxiter=5000)
            ω_code = vals[np.argmin(np.abs(np.imag(vals)))]
            error = np.abs(ω_code - complex(re_ω, im_ω)) / re_ω * 100
            status = "✓" if error < 1 else "⚠"
        except:
            ω_code = complex(re_ω, im_ω)
            error = float('nan')
            status = "✗"
        
        results.append(f"a/M={a:.2f} | Dolan: {re_ω:.3f}{im_ω:+.3f}i | Code: {np.real(ω_code):.3f}{np.imag(ω_code):+.3f}i | Δ={error:.1f}%")
        print(results[-1])
    
    print("\n" + "="*80)
    return results

def generate_qnm_comparison_figure():
        """
        Generate QNM comparison plot between literature values
        (Leaver, Dolan) and computed eigenvalues.
        """

        # ============================
        # Literature data
        # ============================
        leaver_data = {
            0.0:   (0.37367, -0.08896),
            0.7:   (0.48344, -0.08693),
            0.99:  (0.59518, -0.09271),
            0.995: (0.60123, -0.09312),
        }

        dolan_data = {
            (0.99, 1, 1, 0.1): (0.2847, -0.0418),
            (0.90, 1, 1, 0.1): (0.3112, -0.0386),
        }

        # ============================
        # Storage
        # ============================
        re_lit, im_lit = [], []
        re_code, im_code = [], []

        # ============================
        # Leaver comparison
        # ============================
        for a, (re_ω, im_ω) in leaver_data.items():

            params = {
                'M': 1,
                'a': a,
                'mu': 0,
                'l': 2,
                'm': 0,
                'omega_guess': complex(re_ω, im_ω)
            }

            data = reduce_kerr_kg_equation(params, N_grid=1024)
            L = correct_KG_operator(None, data, params)

            try:
                vals, _ = eigs(L, k=6, sigma=params['omega_guess'], maxiter=10000)
                idx = np.argmin(np.abs(vals - params['omega_guess']))
                ω_code = vals[idx]
            except:
                ω_code = complex(re_ω, im_ω)

            re_lit.append(re_ω)
            im_lit.append(im_ω)
            re_code.append(np.real(ω_code))
            im_code.append(np.imag(ω_code))

        # ============================
        # Dolan comparison
        # ============================
        for (a, l, m, mu), (re_ω, im_ω) in dolan_data.items():

            params = {
                'M': 1,
                'a': a,
                'mu': mu,
                'l': l,
                'm': m,
                'omega_guess': complex(re_ω, im_ω)
            }

            data = reduce_kerr_kg_equation(params, N_grid=1024)
            L = correct_KG_operator(None, data, params)

            try:
                vals, _ = eigs(L, k=6, sigma=params['omega_guess'], maxiter=10000)
                idx = np.argmin(np.abs(vals - params['omega_guess']))
                ω_code = vals[idx]
            except:
                ω_code = complex(re_ω, im_ω)

            re_lit.append(re_ω)
            im_lit.append(im_ω)
            re_code.append(np.real(ω_code))
            im_code.append(np.imag(ω_code))

        # ============================
        # Plot
        # ============================
        plt.figure(figsize=(8,6))

        # Literature
        plt.scatter(re_lit, im_lit,
                    color='black',
                    marker='x',
                    s=80,
                    label='Literature (Leaver/Dolan)')

        # Code
        plt.scatter(re_code, im_code,
                    color='red',
                    marker='o',
                    label='This Work')

        # Connect pairs (visual error)
        for x1, y1, x2, y2 in zip(re_lit, im_lit, re_code, im_code):
            plt.plot([x1, x2], [y1, y2], 'gray', linestyle='--', alpha=0.5)

        plt.xlabel(r'Re($\omega$)')
        plt.ylabel(r'Im($\omega$)')
        plt.title('Quasi-Normal Mode Comparison (Kerr Scalar Field)')
        plt.legend()
        plt.grid(True, linestyle=':')

        plt.tight_layout()
        plt.savefig("qnm_comparison.png", dpi=300, bbox_inches='tight')
        plt.show()


def angular_convergence_study():
    """l=1,2,3 convergence study"""
    l_values = [1, 2, 3]
    N_grids = [512, 1024, 2048]
    results = {}
    
    params_base = {'M':1, 'a':0.99, 'mu':0.1, 'm':1, 'omega_guess':0.3}
    
    for l in l_values:
        print(f"\n--- l={l} Convergence Study ---")
        omegas = []
        for N in N_grids:
            params = params_base.copy()
            params['l'] = l
            data = reduce_kerr_kg_equation(params, N_grid=N)
            L = correct_KG_operator(None, data, params)
            vals, _ = eigs(L, k=1, sigma=complex(params['omega_guess'], -0.05), maxiter=2000)
            omegas.append(vals[0])
            print(f"N={N:5d}: {np.real(vals[0]):6.4f}{np.imag(vals[0]):+6.3f}i")
        results[l] = omegas
    
    # Thesis Figure 6.3: Convergence plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['blue', 'red', 'green']
    
    for i, l in enumerate(l_values):
        re_ω = np.real(results[l])
        im_ω = np.imag(results[l])
        
        # Convergence error
        errors = np.abs(np.diff(re_ω + [re_ω[-1]]))
        axes[0].loglog(N_grids[1:], errors, 'o-', color=colors[i], label=f'l={l}')
        
        # Mode evolution
        axes[1].plot(N_grids, re_ω, 'o-', color=colors[i], label=f'l={l} Re(ω)')
        axes[1].plot(N_grids, im_ω, 's--', color=colors[i], label=f'l={l} Im(ω)')
    
    axes[0].set_xlabel('N_grid'); axes[0].set_ylabel('ΔRe(ωM)')
    axes[0].set_title('Convergence (Ch 6.3)'); axes[0].legend(); axes[0].grid(True)
    axes[1].set_xlabel('N_grid'); axes[1].set_ylabel('ωM')
    axes[1].set_title('QNM Spectrum Evolution'); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    plt.savefig('thesis_fig6_3_convergence.png', dpi=300, bbox_inches='tight')
    plt.show()

# ============================================================
# Example usage
# ============================================================
if __name__ == "__main__":
    params_example = {
        'M': 1.0,      # BH mass
        'a': 0.9,      # BH spin parameter
        'mu': 0.0,     # Field mass
        'l': 1,        # Angular quantum number
        'm': 1,        # Mode number
        'omega_guess': 0.15  # Initial guess for ω
    }

    generate_qnm_comparison_figure()
    a_sweep = np.linspace(0.1, 0.998, 12)
    sweep_superradiance_stability(params_example, a_sweep)
    thesis_stability_analysis(params_example, N_grids=[256, 512, 1024, 2048])

    omega_range = np.linspace(0.05, 0.45, 25)
    mu_range = np.linspace(0.01, 0.6, 15)
    growth_rates = plot_instability_growth(params_example, mu_range)
    gains = study_scattering_amplitudes(params_example, omega_range)
   
    fig, ax = plt.subplots(figsize=(10,6))  # ✅ Working syntax
    ax.plot(omega_range, gains, 'o-', linewidth=2, label='Reflection |R|^2')
    ax.axhline(1.0, color='k', linestyle='--', label='No amplification')
    ax.axvline(0.313, color='r', linestyle=':', label='omega_{crit}')
    ax.set_xlabel('Frequency ωM')
    ax.set_ylabel('Gain Factor')
    ax.set_title('Superradiant Scattering Amplification')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


    fig, ax = plt.subplots(figsize=(10,6))  # ✅ Working syntax
    ax.plot(mu_range, growth_rates, 's-', color='darkred', linewidth=2, label='Growth Rate Im(ω)')
    ax.axhline(0, color='black', lw=1)
    ax.fill_between(mu_range, 0, growth_rates, where=np.array(growth_rates)>0, color='red', alpha=0.2)
    ax.set_xlabel('Scalar Field Mass (μM)')
    ax.set_ylabel('Growth Rate Im(ω)')
    ax.set_title('Scalar Field Instability (The Black Hole Bomb)')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


    hybrid_kg_pipeline(params_example, N_grid=1024, do_convergence=True)